# 从 4 个 token 到 Multi-Head Attention

**目标**：从可手算的 Q/K/V 到矩阵乘法、causal mask、softmax、多个 heads，最后观察 next-token loss 如何更新参数。你需要基本矩阵乘法和 Python；先预测再执行每个 cell。

本 notebook 是教学玩具，所有向量都是为了便于手算而构造，不是训练好的词向量。CPU 即可；在 `code/attention_lab` 打开，依次运行。训练及测试在 Mac mini 执行。


## 1. 四个 token 的 Q、K、V

假设输入 token 分别是 A、B、C、D。先略去 embedding 和可学习投影，直接指定投影的结果（后面再补）：

$$Q=K=\begin{bmatrix}1&0\\0&1\\1&1\\-1&0\end{bmatrix},\qquad V=\begin{bmatrix}1&0\\0&1\\1&1\\2&0\end{bmatrix}.$$

每一行一个 token，$d_k=d_v=2$。Q 是查找需求，K 是匹配地址，V 是读取内容；它们只是学习到的向量，语义并非硬编码。


In [ ]:
import math
import torch
from attention import CausalMHA, TinyLM

torch.manual_seed(42)
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.], [-1., 0.]])
K = Q.clone()
V = torch.tensor([[1., 0.], [0., 1.], [1., 1.], [2., 0.]])
print("Q", Q.shape, "\n", Q)
print("K", K.shape, "V", V.shape)


## 2. 手算 scores：为什么是 $QK^\top$？

矩阵乘法 $[4,2]\times[2,4]=[4,4]$：第 $i$ 行、第 $j$ 列是 $q_i\cdot k_j$，表示 query $i$ 对 key $j$ 的匹配。比如第二行：

$$q_2=[0,1],\qquad q_2K^\top=[0,1,1,0].$$

第四个 query 的第一项是 $[-1,0]\cdot[1,0]=-1$。请先自己补完所有 16 个分数，再运行。


In [ ]:
raw_scores = Q @ K.T
expected = torch.tensor([[1., 0., 1., -1.],
                         [0., 1., 1., 0.],
                         [1., 1., 2., -1.],
                         [-1., 0., -1., 1.]])
print(raw_scores)
torch.testing.assert_close(raw_scores, expected)


## 3. 缩放与 causal mask

将 scores 除以 $\sqrt{d_k}=\sqrt{2}$，以控制点积随维度增大时的尺度。decoder 的位置 $i$ 不能看未来 $j>i$，所以在 softmax **之前**将未来位置设为 $-\infty$。

例：第二行缩放并 mask 后为 $[0,1/\sqrt2,-\infty,-\infty]$。为什么不能先 softmax 再简单把未来概率设为 0？提示：剩余概率的和会发生什么？


In [ ]:
scaled = raw_scores / math.sqrt(Q.shape[-1])
future = torch.ones(4, 4, dtype=torch.bool).triu(1)
masked = scaled.masked_fill(future, float('-inf'))
print("future mask (1 = forbidden):\n", future.int())
print("masked scores:\n", masked)


## 4. Softmax：第二个 token 实际读取多少？

softmax 沿最后一维，也就是针对每个 query 在可见 keys 上归一化。令 $a=e^{1/\sqrt2}$，第二行变为：

$$[0,1/\sqrt2,-\infty,-\infty]\;\longrightarrow\;[1/(1+a),a/(1+a),0,0].$$

这里大约是 $[0.3302,0.6698,0,0]$。请先猜第一行会是什么，再运行。


In [ ]:
weights = masked.softmax(dim=-1)
a = math.exp(1 / math.sqrt(2))
print(weights)
torch.testing.assert_close(weights[1], torch.tensor([1/(1+a), a/(1+a), 0., 0.]))
torch.testing.assert_close(weights.sum(-1), torch.ones(4))
assert (weights.triu(1) == 0).all()


## 5. 加权读取 V：得到输出

$$Y=AV,\qquad [4,4]\times[4,2]=[4,2].$$

第二个输出只使用前两个 value：

$$y_2=\frac{1}{1+a}[1,0]+\frac{a}{1+a}[0,1]\approx[0.3302,0.6698].$$

请先手算第一个输出，再运行。改变 $V$ 而保持 Q/K 不变时，权重会不会改变？


In [ ]:
output = weights @ V
print(output)
torch.testing.assert_close(output[0], V[0])
torch.testing.assert_close(output[1], torch.tensor([1/(1+a), a/(1+a)]))
changed_V = V.clone()
changed_V[0] *= 3
print("same weights, different output:", (weights @ changed_V)[1])


## 6. 从手工 Q/K/V 到可学习的多头投影

现在输入 $X$ 为 $[B,T,D]$。每个 head 的输出维度 $d_h=D/H$。代码通过三个 $D\times D$ 线性层一次性计算各头投影，再 reshape；每个 head 对应投影矩阵的不同输出列，因此参数并非共享。

注意：`reshape` 和 `transpose` 只是安排数据维度，学习发生在投影权重中。读 [attention.py](attention.py) 的 `CausalMHA.forward`，对照下面输出逐行写出 shape。


In [ ]:
B, T, D, H = 1, 4, 8, 2
x = torch.randn(B, T, D)
mha = CausalMHA(width=D, heads=H)
y, head_weights = mha(x, return_weights=True)
print("x:", x.shape, "heads:", H, "head_dim:", D//H)
print("attention weights [B,H,T,T]:", head_weights.shape)
print("output [B,T,D]:", y.shape)
print("head 0 weights:\n", head_weights[0, 0])


## 7. 因果性实验：改变未来，不应改变过去

保持 A/B/C 相同，只改变 D。预测前三个输出是否变化、第四个输出是否必须变化。后者**不保证一定变化**，取决于参数和输入；前者必须保持不变。


In [ ]:
changed = x.clone()
changed[:, 3] += 10
y_changed = mha(changed)
torch.testing.assert_close(y[:, :3], y_changed[:, :3], atol=1e-6, rtol=1e-5)
print("前 3 个输出最大差值:", (y[:, :3] - y_changed[:, :3]).abs().max().item())
print("最后输出最大差值:", (y[:, 3] - y_changed[:, 3]).abs().max().item())


## 8. 把多头 attention 接入 next-token 训练

`TinyLM` 是一个单层 decoder：token embedding + 可学习位置 embedding → pre-norm MHA + residual → pre-norm MLP + residual → logits。设 5 个词，输入 `[0,1,2,3]`，目标 `[1,2,3,4]`：每个位置预测下一个 token。

实际文本训练在 `train.py`。这里先观察一次 backward 和更新，注意 $W_Q,W_K,W_V$ 都参与梯度下降；loss 下降并非每一步都得到保证。


In [ ]:
model = TinyLM(vocab_size=5, context=4, width=8, heads=2)
ids = torch.tensor([[0, 1, 2, 3]])
targets = torch.tensor([[1, 2, 3, 4]])
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
logits, loss = model(ids, targets)
print("logits [B,T,vocab]:", logits.shape, "loss:", loss.item())
optimizer.zero_grad(set_to_none=True)
loss.backward()
for name in ('q', 'k', 'v'):
    print(name, "gradient norm:", getattr(model.attention, name).weight.grad.norm().item())
before = model.attention.q.weight.detach().clone()
optimizer.step()
print("Q weight changed:", not torch.equal(before, model.attention.q.weight))


## 9. 动手练习

1. 如果第二行 mask 后改成 `[0, 0, -∞, -∞]`，第二个输出是什么？先手算，再改下面的 `exercise_scores` 验证。
2. 若 head 数从 2 改成 4、D 保持 8，单个 head 的维度与 MHA 的 K/V 总宽度是多少？
3. 对比 [qwen_next_lab/model.py](../qwen_next_lab/model.py)：标出纯 MHA 路径外额外加入的 norm、RoPE、gate。

**答案提示**：问题 1 的两个可见权重相同；问题 2 要区分“每头维度”和“全部头加总”。


In [ ]:
exercise_scores = torch.tensor([0., 0., float('-inf'), float('-inf')])
exercise_output = exercise_scores.softmax(-1) @ V
print("exercise output:", exercise_output)
torch.testing.assert_close(exercise_output, torch.tensor([0.5, 0.5]))


## 下一步与限制

运行 `python -m unittest -v` 后，用 `python train.py --device cpu --steps 300` 做受控小文本过拟合，查看 loss、梯度与生成样本。CPU 跑通后再试 MPS。训练脚本重复使用同一小段文本，不代表验证集泛化。

**本项目没有 KV cache**：生成时对截断窗口重新算 prefix。等你理解这个闭环，再实现 cache 并和 full-prefix logits 做一致性比较；不要用当前代码测量 KV cache 加速。公式遵循 [Transformer 原论文](https://arxiv.org/abs/1706.03762) 的 scaled dot-product attention。
